# Import libraries

In [ ]:
import torch
from transformers import AutoTokenizer
from TweetNormalizer import normalizeTweet
import pandas as pd
import torch.nn as nn
from transformers import AutoModel
import pickle

# Prepare inputs and labels

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
def create_tweets_embeddings(tweets):
    tweets = [normalizeTweet(tweet) for tweet in tweets]
    tweets = tokenizer(tweets, padding=True, truncation=True, return_tensors="pt")
    return tweets["input_ids"].clone().detach(), tweets["attention_mask"]

In [ ]:
def create_label_encodings(labels): #change function name
    # change labels to numbers
    # 0: Demonstrations, 1: Political violence, 2: No relevant event, 3: Strategic developments
    labels = [0 if label == "Demonstrations" else
            1 if label == "Political violence" else
            2 if label == "No relevant event" else
            3 if label == "Strategic developments" else
            None for label in labels]

    # create tensor of labels
    labels = torch.tensor(labels, dtype=torch.long)

    return labels

In [ ]:
def features_to_tensor(features):
    return torch.tensor(features, dtype=torch.float32)

In [ ]:
def dataset_to_embeddings(file_path):
    df = pd.read_csv(file_path)
    tweets = df["text"].tolist()
    labels = df["disorder_type"].tolist()
    features = df[["sentiment_pos","sentiment_neg","hs_hateful","hs_targeted","hs_aggressive","emo_anger",
                   "emo_fear","emo_disgust","emo_sadness","emo_surprise","sentiment_neu"]].values.tolist()

    tweets_embeddings, attention_mask = create_tweets_embeddings(tweets)
    labels = create_label_encodings(labels) #change function name
    features = features_to_tensor(features).float()

    return (tweets_embeddings, features, attention_mask), labels

In [ ]:
# transform data to numerical representations
inputs, labels = dataset_to_embeddings("train.csv")
text_input_ids = inputs[0]
numerical_features = inputs[1]
attention_masks = inputs[2]

## Create embeddings

In [ ]:
bertweet = AutoModel.from_pretrained("distilbert/distilbert-base-uncased")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

In [ ]:
def create_numerical_embeddings(numerical_feature_dim, hidden_dim, numerical_features):
    fc_numerical = nn.Sequential(
      nn.Linear(numerical_feature_dim, hidden_dim),
      nn.ReLU(),
      nn.Dropout(0.3)
    )

    embeddings = fc_numerical(numerical_features)

    return embeddings

In [ ]:
# Get text embeddings from BERT
text_embeddings = bertweet(input_ids=text_input_ids, attention_mask=attention_masks).last_hidden_state

# Pool the embeddings (use the [CLS] token representation or mean pooling)
text_embeddings = text_embeddings.mean(dim=1)

In [ ]:
text_embeddings.shape

torch.Size([4919, 768])

In [ ]:
hidden_dim = 128
numerical_feature_dim = 11

# Process numerical features
numerical_embeddings = create_numerical_embeddings(numerical_feature_dim, hidden_dim, numerical_features)

In [ ]:
numerical_embeddings.shape

torch.Size([4919, 128])

In [ ]:
# Concatenate text and numerical embeddings
combined_embeddings = torch.cat((text_embeddings, numerical_embeddings), dim=1)
print(combined_embeddings.shape)

torch.Size([4919, 896])


# Custom Dataset

In [ ]:
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, inputs, labels):
        self.inputs = inputs
        self.labels = labels

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return {
            'inputs': self.inputs[idx],
            'labels': self.labels[idx]
        }

# Neural Network

In [ ]:
class CombinedModel(nn.Module):
    def __init__(self, hidden_dim, output_dim, input_dim, num_hidden_layers):
        '''
        A fully connected model that combines text embeddings from BERTweet with numerical features to predict the disorder type.

        Args:
            input_dim (int): Dimension of the input features.
            hidden_dim (int): Dimension of the hidden layers.
            num_hidden_layers (int): Number of hidden layers.
            output_dim (int): Dimension of the output layer.
        '''
        super(CombinedModel, self).__init__()

        # Fully connected layers for combined features
        self.fc_combined = nn.Sequential()
        self.fc_combined.append(nn.Linear(input_dim, hidden_dim))
        self.fc_combined.append(nn.ReLU())
        self.fc_combined.append(nn.Dropout(0.3))
        for _ in range(num_hidden_layers - 1):  # Subtract 1 to account for the first layer
            self.fc_combined.append(nn.Linear(hidden_dim, hidden_dim))
            self.fc_combined.append(nn.ReLU())
            self.fc_combined.append(nn.Dropout(0.3))
        self.fc_combined.append(nn.Linear(hidden_dim, output_dim))  # Output layer

    def forward(self, combined_embeddings):
        '''
        Forward pass of the model.
        '''
        # Process combined embeddings
        output = self.fc_combined(combined_embeddings)

        return output

In [ ]:
# Define model parameters
hidden_dim = 128
input_dim = combined_embeddings.shape[1]  # Number of input features
output_dim = 4  # Number of classes for disorder type classification

# Training model


In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from torch.utils.data import TensorDataset
import torch.nn.functional as F

In [ ]:
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score

In [ ]:
class Estimator(BaseEstimator, TransformerMixin):
    def __init__(self, input_dim=896,
                 hidden_dim=128, output_dim=4, num_epochs=10, lr=0.001, batch_size=32, num_hidden_layers=2):
      # consider adding number of hidden layers
      self.input_dim = input_dim
      self.hidden_dim = hidden_dim
      self.output_dim = output_dim
      self.num_epochs = num_epochs
      self.lr = lr
      self.batch_size = batch_size
      self.num_hidden_layers = num_hidden_layers
      self.model = CombinedModel(self.hidden_dim, self.output_dim, self.input_dim, self.num_hidden_layers)
      self.criterion = nn.CrossEntropyLoss()
      self.optimizer = torch.optim.Adam(self.model.parameters(), lr=self.lr)

    def fit(self, X, y):
      X = torch.tensor(X)
      y = torch.tensor(y)
      dataset = TensorDataset(X, y)
      dataloader = DataLoader(dataset, batch_size=self.batch_size, shuffle=True)
      for epoch in range(self.num_epochs):
        self.model.train()
        running_loss = 0.0

        for i, (inputs, labels) in enumerate(dataloader):
            self.optimizer.zero_grad()
            outputs = self.model(inputs)
            loss =self.criterion(outputs, labels)
            loss.backward()
            self.optimizer.step()

            running_loss += loss.item()

        epoch_loss = running_loss / len(dataloader)
        print(f'Epoch [{epoch+1}/{self.num_epochs}], Loss: {epoch_loss:.4f}')

      return self

    def predict(self, X):
      X = torch.tensor(X)
      self.model.eval()
      with torch.no_grad():
        outputs = self.model(X)
        probabilities = F.softmax(outputs, dim=1)  # Get probabilities
        _, predicted = torch.max(outputs.data, 1)
        return predicted.cpu().numpy()

    def predict_proba(self, X):
      X = torch.tensor(X)
      self.model.eval()
      with torch.no_grad():
          outputs = self.model(X)  # Assuming attention_mask is None
          probabilities = F.softmax(outputs, dim=1)
      return probabilities.numpy()

## With Grid Search

In [ ]:
param_grid = {
    'hidden_dim': [64, 128],
    'num_hidden_layers': [1, 2, 3],
    'lr': [1e-4, 1e-3],
    'num_epochs': [5, 10],
    'batch_size': [16, 32]
}

param_grid_two = {
    'hidden_dim': [64, 128, 256],
    'num_hidden_layers': [1, 2, 3, 4],
    'lr': [1e-2, 1e-3, 1e-4, 1e-5],
    'num_epochs': [5, 10, 12],
    'batch_size': [16, 32, 64]
}

In [ ]:
# Define the scoring dictionary
scoring = {
    'accuracy': 'accuracy',
    'precision_macro': make_scorer(precision_score, average='macro', zero_division = 0),
    'recall_macro': make_scorer(recall_score, average='macro', zero_division = 0),
    'f1_macro': make_scorer(f1_score, average='macro', zero_division = 0),
    'precision_weighted': make_scorer(precision_score, average='weighted', zero_division = 0),
    'recall_weighted': make_scorer(recall_score, average='weighted', zero_division = 0),
    'f1_weighted': make_scorer(f1_score, average='weighted', zero_division = 0),
}

In [ ]:
model = Estimator()
grid_search = GridSearchCV(model, param_grid_two, cv=5, scoring=scoring, refit='f1_weighted')

In [ ]:
print(combined_embeddings.shape)
print(labels.shape)

torch.Size([4919, 896])
torch.Size([4919])


In [ ]:
combined_embeddings = combined_embeddings.detach().numpy()
labels = labels.long().detach().numpy()

In [ ]:
grid_search.fit(combined_embeddings, labels)

Epoch [1/5], Loss: 1.0626
Epoch [2/5], Loss: 0.9758
Epoch [3/5], Loss: 0.9174
Epoch [4/5], Loss: 0.8858
Epoch [5/5], Loss: 0.8591
Epoch [1/5], Loss: 1.0678
Epoch [2/5], Loss: 0.9727


KeyboardInterrupt: 

In [ ]:
best_params = grid_search.best_params_
print(best_params)

In [ ]:
# print(torch.isnan(combined_embeddings).any())
# print(torch.isnan(labels).any())

In [ ]:
print("Best score: ", grid_search.best_score_)

In [ ]:
# Get all evaluation results
results = grid_search.cv_results_
print(results)

In [ ]:
best_model = grid_search.best_estimator_

In [ ]:
# save the model
filename = 'disorder_type_model.pkl'
pickle.dump(best_model, open(filename, 'wb'))

## With k-fold

In [ ]:
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
disorder_model = Estimator(input_dim=896, hidden_dim=128, output_dim=4, num_epochs=10, lr=0.001, batch_size=16, num_hidden_layers=1)

accuracies = []
precisions_macro = []
recalls_macro = []
f1s_macro = []
precisions_weighted = []
recalls_weighted = []
f1s_weighted = []

for train_index, val_index in kf.split(combined_embeddings):
    X_train, X_val = combined_embeddings[train_index], combined_embeddings[val_index]
    y_train, y_val = labels[train_index], labels[val_index]

    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)

    accuracy = accuracy_score(y_val, y_pred)
    precision_macro = precision_score(y_val, y_pred, average='macro')
    recall_macro = recall_score(y_val, y_pred, average='macro')
    f1_macro = f1_score(y_val, y_pred, average='macro')
    precision_weighted = precision_score(y_val, y_pred, average='weighted')
    recall_weighted = recall_score(y_val, y_pred, average='weighted')
    f1_weighted = f1_score(y_val, y_pred, average='weighted')

    accuracies.append(accuracy)
    precisions_macro.append(precision_macro)
    recalls_macro.append(recall_macro)
    f1s_macro.append(f1_macro)
    precisions_weighted.append(precision_weighted)
    recalls_weighted.append(recall_weighted)
    f1s_weighted.append(f1_weighted)

    print(f'Fold accuracy: {accuracy}')
    print(f'Fold precision (macro): {precision_macro}')
    print(f'Fold recall (macro): {recall_macro}')
    print(f'Fold F1 score (macro): {f1_macro}')
    print(f'Fold precision (weighted): {precision_weighted}')
    print(f'Fold recall (weighted): {recall_weighted}')
    print(f'Fold F1 score (weighted): {f1_weighted}')

average_accuracy = np.mean(accuracies)
average_precision_macro = np.mean(precisions_macro)
average_recall_macro = np.mean(recalls_macro)
average_f1_macro = np.mean(f1s_macro)
average_precision_weighted = np.mean(precisions_weighted)
average_recall_weighted = np.mean(recalls_weighted)
average_f1_weighted = np.mean(f1s_weighted)

print(f'Average accuracy: {average_accuracy}')
print(f'Average precision (macro): {average_precision_macro}')
print(f'Average recall (macro): {average_recall_macro}')
print(f'Average F1 score (macro): {average_f1_macro}')
print(f'Average precision (weighted): {average_precision_weighted}')
print(f'Average recall (weighted): {average_recall_weighted}')
print(f'Average F1 score (weighted): {average_f1_weighted}')


Epoch [1/12], Loss: 1.0754
Epoch [2/12], Loss: 0.9880
Epoch [3/12], Loss: 0.9405
Epoch [4/12], Loss: 0.9058
Epoch [5/12], Loss: 0.8791
Epoch [6/12], Loss: 0.8455
Epoch [7/12], Loss: 0.8234
Epoch [8/12], Loss: 0.7944
Epoch [9/12], Loss: 0.7741
Epoch [10/12], Loss: 0.7465
Epoch [11/12], Loss: 0.7218
Epoch [12/12], Loss: 0.6963
Fold accuracy: 0.6097560975609756
Fold precision (macro): 0.4258441190919226
Fold recall (macro): 0.4228513730415549
Fold F1 score (macro): 0.4199259831829638
Fold precision (weighted): 0.6005520945621982
Fold recall (weighted): 0.6097560975609756
Fold F1 score (weighted): 0.5979905465462156


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Epoch [1/12], Loss: 0.7499
Epoch [2/12], Loss: 0.7081
Epoch [3/12], Loss: 0.6708
Epoch [4/12], Loss: 0.6503
Epoch [5/12], Loss: 0.6177
Epoch [6/12], Loss: 0.5923
Epoch [7/12], Loss: 0.5715
Epoch [8/12], Loss: 0.5483
Epoch [9/12], Loss: 0.5263
Epoch [10/12], Loss: 0.4935
Epoch [11/12], Loss: 0.4772
Epoch [12/12], Loss: 0.4480
Fold accuracy: 0.6880081300813008
Fold precision (macro): 0.4903204012080803
Fold recall (macro): 0.47512930871748404
Fold F1 score (macro): 0.4803584243787069
Fold precision (weighted): 0.674711276080075
Fold recall (weighted): 0.6880081300813008
Fold F1 score (weighted): 0.6790313265574411


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Epoch [1/12], Loss: 0.5931
Epoch [2/12], Loss: 0.5173
Epoch [3/12], Loss: 0.4829
Epoch [4/12], Loss: 0.4765
Epoch [5/12], Loss: 0.4503
Epoch [6/12], Loss: 0.4298
Epoch [7/12], Loss: 0.4134
Epoch [8/12], Loss: 0.4080
Epoch [9/12], Loss: 0.3878
Epoch [10/12], Loss: 0.3862
Epoch [11/12], Loss: 0.3693
Epoch [12/12], Loss: 0.3450
Fold accuracy: 0.7855691056910569
Fold precision (macro): 0.5715101247003228
Fold recall (macro): 0.5843342387460034
Fold F1 score (macro): 0.57767540913532
Fold precision (weighted): 0.7773334365771446
Fold recall (weighted): 0.7855691056910569
Fold F1 score (weighted): 0.78129429072289
Epoch [1/12], Loss: 0.4489
Epoch [2/12], Loss: 0.3806
Epoch [3/12], Loss: 0.3922
Epoch [4/12], Loss: 0.3497
Epoch [5/12], Loss: 0.3473
Epoch [6/12], Loss: 0.3203
Epoch [7/12], Loss: 0.3098
Epoch [8/12], Loss: 0.3122
Epoch [9/12], Loss: 0.3103
Epoch [10/12], Loss: 0.3063
Epoch [11/12], Loss: 0.2729
Epoch [12/12], Loss: 0.2794
Fold accuracy: 0.8607723577235772
Fold precision (macro):

In [ ]:
# save the model
filename = 'best_disorer_type_model.pkl'
pickle.dump(model, open(filename, 'wb'))